# Ligand Preparation (Stage 0.1.4)

Prepare the ligand for molecular docking.

## Preliminaries

In [1]:
from os.path import abspath

import meeko
import molscrub
import pandas as pd
from rdkit import Chem

from src.utils.ligprep import LigPrep
from src.utils.logger import Logger
from src.utils.housekeeping import (
    fix_dir,
    log_pkg_ver,
    remove_prev_data
)


In [2]:
log = Logger("stage-0.1.4-ligand_prep")


INFO     Setup of logger complete, started <stage-0.1.4-ligand_prep> 2026-06-04 02:51:10.521689

In [3]:
pkg_imports: list[str] = ["meeko", "molscrub", "pandas", "rdkit"]
log_pkg_ver(pkg_imports, log)


INFO     Python v 3.10.20

INFO     Import meeko v 0.7.1

INFO     Import molscrub v 0.2.2

INFO     Import pandas v 2.3.3

INFO     Import rdkit v 2025.9.5

In [4]:
DIR: list[str] = [
        "../data/ligands/prepped",
        "../data/ligands/prepped/scrub",
        "../data/mirror_ligands/prepped",
        "../data/mirror_ligands/prepped/scrub"
    ]


In [5]:
remove_prev_data(DIR, log)


In [6]:
fix_dir(DIR, log)


INFO     ../data/ligands/prepped is missing, include to the list ...

INFO     ../data/ligands/prepped/scrub is missing, include to the list ...

INFO     ../data/mirror_ligands/prepped is missing, include to the list ...

INFO     ../data/mirror_ligands/prepped/scrub is missing, include to the list ...

INFO     Trying to create dir: ../data/ligands/prepped

INFO     Trying to create dir: ../data/ligands/prepped/scrub

INFO     Trying to create dir: ../data/mirror_ligands/prepped

INFO     Trying to create dir: ../data/mirror_ligands/prepped/scrub

In [7]:
CWD: str = abspath(".")
MIR_DIR: str = f"{CWD}/../data/mirror_ligands/prepped"
NAT_DIR: str = f"{CWD}/../data/ligands/prepped"


## Data

In [8]:
native = pd.read_csv(
        f"../data//natural_antibiotics.csv", index_col=0
    )
mirror = pd.read_csv(f"../data/mirror_ligand.csv", index_col=0)


In [9]:
native.head()


,Amoxicillin,Piperacillin,Ticarcillin,Cefepime,Cefazolin,Ceftolozane,Ceftriaxone,Ceftazidime,Aztreonam,Imipenem,...,Fosfomycin,Amikacin,Tobramycin,Streptomycin,Kanamycin,Trimethoprim,Sulfamethoxazole,Ciprofloxacin,Levofloxacin,Moxifloxacin
cid,33613,43672,36921,5479537,33255,53234134,5479530,5481173,5742832,104838,...,446987,37768,36294,19649,6032,5578,5329,2764,149096,152946
smiles,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,CCN1CCN(C(=O)C1=O)C(=O)N[C@H](C2=CC=CC=C2)C(=O...,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,C[N+]1(CCCC1)CC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC1=NN=C(S1)SCC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC(C)(C(=O)O)O/N=C(/C1=NSC(=N1)N)\C(=O)N[C@H]2...,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...,CC(C)(C(=O)O)O/N=C(/C1=CSC(=N1)N)\C(=O)N[C@H]2...,C[C@H]1[C@@H](C(=O)N1S(=O)(=O)O)NC(=O)/C(=N\OC...,C[C@H]([C@@H]1[C@H]2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O,...,C[C@H]1[C@H](O1)P(=O)(O)O,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1N)O[C@@H]2[...,C[C@H]1[C@@]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@...,C1[C@H]([C@@H]([C@H]([C@@H]([C@H]1N)O[C@@H]2[C...,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N,CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,COC1=C2C(=CC(=C1N3C[C@@H]4CCCN[C@@H]4C3)F)C(=O...
inchi,InChI=1S/C16H19N3O5S/c1-16(2)11(15(23)24)19-13...,InChI=1S/C23H27N5O7S/c1-4-26-10-11-27(19(32)18...,InChI=1S/C15H16N2O6S2/c1-15(2)9(14(22)23)17-11...,InChI=1S/C19H24N6O5S2/c1-25(5-3-4-6-25)7-10-8-...,InChI=1S/C14H14N8O4S3/c1-6-17-18-14(29-6)28-4-...,"InChI=1S/C23H30N12O8S2/c1-23(2,20(40)41)43-31-...",InChI=1S/C18H18N8O7S3/c1-25-18(22-12(28)13(29)...,"InChI=1S/C22H22N6O7S2/c1-22(2,20(33)34)35-26-1...","InChI=1S/C13H17N5O8S2/c1-5-7(10(20)18(5)28(23,...",InChI=1S/C12H17N3O4S/c1-6(16)9-7-4-8(20-3-2-14...,...,"InChI=1S/C3H7O4P/c1-2-3(7-2)8(4,5)6/h2-3H,1H3,...",InChI=1S/C22H43N5O13/c23-2-1-8(29)20(36)27-7-3...,InChI=1S/C18H37N5O9/c19-3-9-8(25)2-7(22)17(29-...,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",InChI=1S/C18H36N4O11/c19-2-6-10(25)12(27)13(28...,InChI=1S/C14H18N4O3/c1-19-10-5-8(6-11(20-2)12(...,InChI=1S/C10H11N3O3S/c1-7-6-10(12-16-7)13-17(1...,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,InChI=1S/C18H20FN3O4/c1-10-9-26-17-14-11(16(23...,InChI=1S/C21H24FN3O4/c1-29-20-17-13(19(26)14(2...


In [10]:
mirror.head()


,Amoxicillin,Piperacillin,Ticarcillin,Cefepime,Cefazolin,Ceftolozane,Ceftriaxone,Ceftazidime,Aztreonam,Imipenem,...,Fosfomycin,Amikacin,Tobramycin,Streptomycin,Kanamycin,Trimethoprim,Sulfamethoxazole,Ciprofloxacin,Levofloxacin,Moxifloxacin
cid,33613,43672,36921,5479537,33255,53234134,5479530,5481173,5742832,104838,...,446987,37768,36294,19649,6032,5578,5329,2764,149096,152946
smiles,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,CCN1CCN(C(=O)C1=O)C(=O)N[C@H](C2=CC=CC=C2)C(=O...,CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H]...,C[N+]1(CCCC1)CC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC1=NN=C(S1)SCC2=C(N3[C@@H]([C@@H](C3=O)NC(=O)...,CC(C)(C(=O)O)O/N=C(/C1=NSC(=N1)N)\C(=O)N[C@H]2...,CN1C(=NC(=O)C(=O)N1)SCC2=C(N3[C@@H]([C@@H](C3=...,CC(C)(C(=O)O)O/N=C(/C1=CSC(=N1)N)\C(=O)N[C@H]2...,C[C@H]1[C@@H](C(=O)N1S(=O)(=O)O)NC(=O)/C(=N\OC...,C[C@H]([C@@H]1[C@H]2CC(=C(N2C1=O)C(=O)O)SCCN=CN)O,...,C[C@H]1[C@H](O1)P(=O)(O)O,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1NC(=O)[C@H]...,C1[C@@H]([C@H]([C@@H]([C@H]([C@@H]1N)O[C@@H]2[...,C[C@H]1[C@@]([C@H]([C@@H](O1)O[C@@H]2[C@H]([C@...,C1[C@H]([C@@H]([C@H]([C@@H]([C@H]1N)O[C@@H]2[C...,COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N,CC1=CC(=NO1)NS(=O)(=O)C2=CC=C(C=C2)N,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,COC1=C2C(=CC(=C1N3C[C@@H]4CCCN[C@@H]4C3)F)C(=O...
inchi,InChI=1S/C16H19N3O5S/c1-16(2)11(15(23)24)19-13...,InChI=1S/C23H27N5O7S/c1-4-26-10-11-27(19(32)18...,InChI=1S/C15H16N2O6S2/c1-15(2)9(14(22)23)17-11...,InChI=1S/C19H24N6O5S2/c1-25(5-3-4-6-25)7-10-8-...,InChI=1S/C14H14N8O4S3/c1-6-17-18-14(29-6)28-4-...,"InChI=1S/C23H30N12O8S2/c1-23(2,20(40)41)43-31-...",InChI=1S/C18H18N8O7S3/c1-25-18(22-12(28)13(29)...,"InChI=1S/C22H22N6O7S2/c1-22(2,20(33)34)35-26-1...","InChI=1S/C13H17N5O8S2/c1-5-7(10(20)18(5)28(23,...",InChI=1S/C12H17N3O4S/c1-6(16)9-7-4-8(20-3-2-14...,...,"InChI=1S/C3H7O4P/c1-2-3(7-2)8(4,5)6/h2-3H,1H3,...",InChI=1S/C22H43N5O13/c23-2-1-8(29)20(36)27-7-3...,InChI=1S/C18H37N5O9/c19-3-9-8(25)2-7(22)17(29-...,"InChI=1S/C21H39N7O12/c1-5-21(36,4-30)16(40-17-...",InChI=1S/C18H36N4O11/c19-2-6-10(25)12(27)13(28...,InChI=1S/C14H18N4O3/c1-19-10-5-8(6-11(20-2)12(...,InChI=1S/C10H11N3O3S/c1-7-6-10(12-16-7)13-17(1...,InChI=1S/C17H18FN3O3/c18-13-7-11-14(8-15(13)20...,InChI=1S/C18H20FN3O4/c1-10-9-26-17-14-11(16(23...,InChI=1S/C21H24FN3O4/c1-29-20-17-13(19(26)14(2...
mirror_smi,CC1(C)S[C@H]2[C@@H](NC(=O)[C@@H](N)c3ccc(O)cc3...,CCN1CCN(C(=O)N[C@H](C(=O)N[C@H]2C(=O)N3[C@H]2S...,CC1(C)S[C@H]2[C@@H](NC(=O)[C@@H](C(=O)O)c3ccsc...,CO/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)[O-])=C(C[N+...,Cc1nnc(SCC2=C(C(=O)O)N3C(=O)[C@H](NC(=O)Cn4cnn...,Cn1c(N)c(NC(=O)NCCN)c[n+]1CC1=C(C(=O)[O-])N2C(...,CO/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)O)=C(CSc3nc(...,CC(C)(O/N=C(\C(=O)N[C@H]1C(=O)N2C(C(=O)[O-])=C...,C[C@@H]1[C@@H](NC(=O)/C(=N\OC(C)(C)C(=O)O)c2cs...,C[C@H](O)[C@@H]1C(=O)N2C(C(=O)O)=C(SCCN=CN)C[C...,...,C[C@H]1O[C@H]1P(=O)(O)O,NCC[C@@H](O)C(=O)N[C@H]1C[C@@H](N)[C@H](O[C@@H...,NC[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O[C@@H...,CN[C@H]1[C@@H](O[C@@H]2[C@@H](O[C@@H]3[C@@H](O...,NC[C@@H]1O[C@@H](O[C@@H]2[C@@H](O)[C@H](O[C@@H...,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1,O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O,C[C@@H]1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O...,COc1c(N2C[C@H]3CCCN[C@H]3C2)c(F)cc2c(=O)c(C(=O...


## Ligand Preparation

In [11]:
ligprep: LigPrep = LigPrep(log)


In [12]:
prepped_mir: list[str] = []
prepped_nat: list[str] = []

for nat, mir in zip(native, mirror):
    nat_prep: str | None = ligprep.ligand_prep(
            native[nat]["smiles"],
            f"Native-{native[nat]['cid']}",
            NAT_DIR
        )
    mir_prep: str | None = ligprep.ligand_prep(
            mirror[mir]["mirror_smi"],
            f"Mirror-{mirror[mir]['cid']}",
            MIR_DIR
        )

    if nat_prep:
        prepped_nat.append(nat_prep)

    if mir_prep:
        prepped_mir.append(mir_prep)


INFO     Scrubing Native-33613 with scrub.py                                                                       
         CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)[C@@H](C3=CC=C(C=C3)O)N)C(=O)O)C -o                               
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         ligands/prepped/scrub/Native-33613-scrub.sdf --ph 7.4 --skip_tautomer

Scrub completed.
Summary of what happened:
Input molecules supplied: 1
mols processed: 1, skipped by rdkit: 0, failed: 0
nr isomers (tautomers and acid/base conjugates): 1 (avg. 1.000 per mol)
nr conformers:  1 (avg. 1.000 per isomer, 1.000 per mol)


INFO     Running mk_prepapre_ligand.py on Native-33613 with mk_prepare_ligand.py -i                                
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         ligands/prepped/scrub/Native-33613-scrub.sdf -o                                                           
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         ligands/prepped/Native-33613-prepped.pdbqt

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


INFO     Scrubing Mirror-33613 with scrub.py CC1(C)S[C@H]2[C@@H](NC(=O)[C@@H](N)c3ccc(O)cc3)C(=O)N2[C@@H]1C(=O)O -o
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         mirror_ligands/prepped/scrub/Mirror-33613-scrub.sdf --ph 7.4 --skip_tautomer

...

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


INFO     Scrubing Mirror-152946 with scrub.py COc1c(N2C[C@H]3CCCN[C@H]3C2)c(F)cc2c(=O)c(C(=O)O)cn(C3CC3)c12 -o     
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         mirror_ligands/prepped/scrub/Mirror-152946-scrub.sdf --ph 7.4 --skip_tautomer

Scrub completed.
Summary of what happened:
Input molecules supplied: 1
mols processed: 1, skipped by rdkit: 0, failed: 0
nr isomers (tautomers and acid/base conjugates): 1 (avg. 1.000 per mol)
nr conformers:  1 (avg. 1.000 per isomer, 1.000 per mol)


INFO     Running mk_prepapre_ligand.py on Mirror-152946 with mk_prepare_ligand.py -i                               
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         mirror_ligands/prepped/scrub/Mirror-152946-scrub.sdf -o                                                   
         /var/home/aeersriv/Documents/FLD-1-antibiotics_mirror_life/antibiotics_mirror_life-CODE/notebooks/../data/
         mirror_ligands/prepped/Mirror-152946-prepped.pdbqt

Input molecules processed: 1, skipped: 0
PDBQT files written: 1
PDBQT files not written due to error: 0
Input molecules with errors: 0


In [13]:
print(
    f"Prepared native ligands: {len(prepped_nat)}\n"
    f"Prepared mirror ligands: {len(prepped_mir)}\n"
)


Prepared native ligands: 34
Prepared mirror ligands: 34

